In [ ]:
!pip install pinecone==10.0.0 pypdf==6.9.1 openai==2.29.0

In [ ]:
import os
import numpy as np
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader
from getpass import getpass
import io
import requests


In [ ]:
# from dotenv import load_dotenv
# load_dotenv()
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")  # Ensure this is in your .env

OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ")

# Initialize clients
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# Test the connection with a lightweight model call
try:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Ping"}],
        max_tokens=5
    )
    print("✅ Connection successful! Response:", response.choices[0].message.content.strip())

except Exception as e:
    print("❌ Connection failed!")
    print(f"Error details: {e}")


In [ ]:
PINECONE_API_KEY = getpass("Enter your Pinecone API Key: ")

# Initialize clients
pc_client = Pinecone(api_key=PINECONE_API_KEY)

# Test the connection by listing your existing indexes
try:
    active_indexes = pc_client.list_indexes()
    print("✅ Connection successful! Authenticated with Pinecone.")
    print(f"Available indexes: {[idx.name for idx in active_indexes]}")
    
except Exception as e:
    print("❌ Connection failed!")
    print(f"Error details: {e}")


## Define Index Configurations

In [6]:
INDEX_NAME = "pdf-rag-index"
DIMENSION = 1536  # Dimension for 'text-embedding-3-small'

## Step 1) Ingestion phase
- Here we populate the vector DB

### Read PDF

In [ ]:
def read_pdf(file_source):
    
    # Check if the input is a web URL
    if file_source.startswith("http://") or file_source.startswith("https://"):
        response = requests.get(file_source)
        response.raise_for_status()  # Check for download errors
        file_object = io.BytesIO(response.content)
    else:
        # Otherwise, treat it as a local file path
        file_object = file_source

    reader = PdfReader(file_object)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text


In [ ]:
# pdf_path = "document-loxford-company.pdf" # On local machine
pdf_path = "https://raw.githubusercontent.com/ash322ash422/data/main/pdf/document-loxford-company.pdf"

pdf_text = read_pdf(pdf_path)

print(pdf_text)

Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 2012, 
the company has grown rapidly into a mid-sized enterprise serving clients across 
North America, Europe, and Asia. 
Loxford Technologies was founded by Daniel Reeves, a former data scientist, and 
Anika Lomax, a software engineer with a background in distributed systems. The 
founders envisioned a company that could bridge the gap between cutting-edge 
AI research and real-world business applications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax 
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK 
• Bengaluru, India 
• Berlin, Germany 
As of 2024, Loxford Technologies reported an estimated annual revenue of $180 
million, with a year-over-year growth rate of approximately 22%. The company 
employs over 850 professionals, including engineers, d

### CHUNK TEXT


In [ ]:
def chunk_text(text, size=80):
    # Split the text by whitespace into individual words
    words = text.split()
    
    # Group words into chunks of the specified size and rejoin them with spaces
    return [" ".join(words[i : i + size]) for i in range(0, len(words), size)]


In [24]:
pdf_chunks = chunk_text(pdf_text)

for i,chunk in enumerate(pdf_chunks):
    print(f"Chunk {i+1}: \n{chunk}")
    print("--------------------\n")

Chunk 1: 
Loxford Technologies – Company Overview Loxford Technologies is a global technology solutions provider specializing in artificial intelligence, data analytics, and enterprise automation. Founded in 2012, the company has grown rapidly into a mid-sized enterprise serving clients across North America, Europe, and Asia. Loxford Technologies was founded by Daniel Reeves, a former data scientist, and Anika Lomax, a software engineer with a background in distributed systems. The founders envisioned a company that could bridge the gap between cutting-edge AI research and
--------------------

Chunk 2: 
real-world business applications. • CEO: Daniel Reeves • CTO: Anika Lomax The company is headquartered in Austin, Texas, USA, with additional offices in: • London, UK • Bengaluru, India • Berlin, Germany As of 2024, Loxford Technologies reported an estimated annual revenue of $180 million, with a year-over-year growth rate of approximately 22%. The company employs over 850 professional

In [ ]:
# Practise for students:
# Later on try following chunking that has overlapping

# def chunk_text(text, size=80, overlap=10):
#     # Split the text by whitespace into individual words
#     words = text.split()
    
#     chunks = []
#     # Step forward by (size - overlap) to create the sliding window
#     step = size - overlap
    
#     # Ensure step is at least 1 to avoid an infinite loop
#     if step <= 0:
#         raise ValueError("Chunk size must be greater than overlap.")
        
#     for i in range(0, len(words), step):
#         chunk = " ".join(words[i : i + size])
#         chunks.append(chunk)
        
#         # Optional: Stop if the current chunk reached the end of the text
#         if i + size >= len(words):
#             break
            
#     return chunks

# Example usage:
# pdf_chunks = chunk_text(pdf_text)

# for i,chunk in enumerate(pdf_chunks):
#     print(f"Chunk {i+1}: \n{chunk}")
#     print("--------------------\n")

### Embeddings

In [ ]:
def embed(texts):
    response = openai_client.embeddings.create(model="text-embedding-3-small",
                                                input=texts
    )
    
    return np.array([d.embedding for d in response.data]).astype("float32")


In [28]:
# Lets convert one chunk into a vector
print(pdf_chunks[0])
print("-----------------")

vector_chunk = embed(pdf_chunks[0])
print(vector_chunk) 

Loxford Technologies – Company Overview Loxford Technologies is a global technology solutions provider specializing in artificial intelligence, data analytics, and enterprise automation. Founded in 2012, the company has grown rapidly into a mid-sized enterprise serving clients across North America, Europe, and Asia. Loxford Technologies was founded by Daniel Reeves, a former data scientist, and Anika Lomax, a software engineer with a background in distributed systems. The founders envisioned a company that could bridge the gap between cutting-edge AI research and
-----------------
[[-0.03085327 -0.00841522  0.01006317 ...  0.02055359 -0.00788879
   0.01026917]]


### Store chunks in vector DB

In [ ]:
# The following function builds the index and stores the chunks into pinecone vector DB

def build_index(chunks):
    # 1. Create index if it doesn't exist
    if INDEX_NAME not in pc_client.list_indexes().names():
        pc_client.create_index(
            name=INDEX_NAME,
            dimension=DIMENSION,
            metric="cosine",  # Cosine is recommended for OpenAI embeddings
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )

    index = pc_client.Index(INDEX_NAME)

    # 2. Generate embeddings
    embeddings = embed(chunks)

    # 3. Format data for Pinecone upsert: (id, vector, metadata)
    vectors_to_upsert = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        v = {"id": f"chunk-{i}", 
             "values": embedding.tolist(), 
             "metadata": {"text": chunk}
            }
        # print("Inserting: ", v) # DO NOT PRINT THIS. IT IS HUGE
        vectors_to_upsert.append(v)

    # 4. Upload to cloud
    index.upsert(vectors = vectors_to_upsert)
    return index


In [45]:
pinecone_index = build_index(pdf_chunks)

print("Upload complete!")


Upload complete!


## Step 2) Retrieval

In [46]:
def retrieve(query, index, k=3):
    q_emb = embed([query])[0].tolist()

    # Query Pinecone and request metadata to get the original text back
    results = index.query(vector=q_emb, top_k=k, include_metadata=True)

    return [match["metadata"]["text"] for match in results["matches"]]

In [47]:
query = "Who was the founder Loxford Tech company?"
print(f"Querying the database for: '{query}'")
print("---------------------------\n")

# Fetch context from Pinecone
retrieved_contexts = retrieve(query, pinecone_index, k=2)
print("retrieved_contexts:", retrieved_contexts)
print("---------------------------\n")

combined_context = "\n".join(retrieved_contexts)
print("combined_context:", combined_context)


Querying the database for: 'Who was the founder Loxford Tech company?'
---------------------------

retrieved_contexts: ['Loxford Technologies – Company Overview Loxford Technologies is a global technology solutions provider specializing in artificial intelligence, data analytics, and enterprise automation. Founded in 2012, the company has grown rapidly into a mid-sized enterprise serving clients across North America, Europe, and Asia. Loxford Technologies was founded by Daniel Reeves, a former data scientist, and Anika Lomax, a software engineer with a background in distributed systems. The founders envisioned a company that could bridge the gap between cutting-edge AI research and', 'real-world business applications. • CEO: Daniel Reeves • CTO: Anika Lomax The company is headquartered in Austin, Texas, USA, with additional offices in: • London, UK • Bengaluru, India • Berlin, Germany As of 2024, Loxford Technologies reported an estimated annual revenue of $180 million, with a year-ov


## ASK LLM - Generation


In [ ]:
def ask_llm(question, context):
    prompt = f"""
    Answer using ONLY the context below. Do NOT make up answers.

    Context:
    {context}

    Question: 
    {question}
    """

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


In [50]:
print("Sending context to GPT-4o-mini...")
answer = ask_llm(query, combined_context)

print("\n--- Final Answer from LLM ---")
print(answer)


Sending context to GPT-4o-mini...

--- Final Answer from LLM ---
The founders of Loxford Technologies are Daniel Reeves and Anika Lomax.


# Delete the index

In [ ]:
# Check and delete the index
if INDEX_NAME in pc_client.list_indexes().names():
    print(f"Deleting index '{INDEX_NAME}'...")
    pc_client.delete_index(INDEX_NAME)
    print("Index deleted successfully!")
else:
    print(f"Index '{INDEX_NAME}' does not exist.")

Deleting index 'pdf-rag-index'...
Index deleted successfully!


# Questions
1) Which embedding model is used here ? 
2) How many dimension does embedding model has ?
3) Which vector DB did we use ?
4) How many chunks did we get from above documents ?
5) How many vector did we store in vector DB ?


# Task for students
- Use your PDF resume
- Switch chunk strategy to overlapping / heading-based
 